## Setup & Configuration

### Imports
- **`sqlite3`** – Connect to and query the local SQLite database.
- **`pandas`** – Load query results into DataFrames for analysis and manipulation.
- **`numpy`** – Support for numerical operations and array handling.
- **`datetime`** – Work with date and time values (e.g., filtering or timestamping).

### Pandas Display Options
- `display.max_columns = 50` → Show up to 50 columns without truncation.
- `display.width = 200` → Widen the console output to prevent line wrapping.
- `display.float_format` → Format floats with thousands separators and 2 decimal places for readability.

### Database Configuration
- **`DB_PATH`** – Path to the SQLite database file (`vault_chain_flow.db`) located in the `customer_data` folder.
- **`TABLE`** – Name of the target table (`Customer_Transactions`) to query.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DB_PATH = r"../customer_data/vault_chain_flow.db"
TABLE = "Customer_Transactions"

### Connect to Database & List Tables

- **`con = sqlite3.connect(DB_PATH)`** – Open a connection to the SQLite database at the configured path.
- **`cursor = con.cursor()`** – Create a cursor object to execute SQL statements.
- **`cursor.execute(...)`** – Run a query against `sqlite_master` (SQLite's internal schema table) to retrieve the names of all tables of type `'table'` in the database.
- **`cursor.fetchall()`** – Fetch all matching rows; prints a list of table names so we can verify the database contents before querying.

In [2]:
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

Tables in database: [('Customer_Transactions',)]


### Load Table into a DataFrame

- **`pd.read_sql_query(f"SELECT * FROM {TABLE}", con)`** – Execute a `SELECT *` query against the `Customer_Transactions` table and load the full result set directly into a pandas DataFrame (`df`).
- **`print(...)`** – Output a summary showing the total number of rows (formatted with thousands separators) and the number of columns, giving a quick confirmation of the data volume loaded.

In [3]:
df = pd.read_sql_query(f"SELECT * FROM Customer_Transactions", con)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

Loaded 580,962 rows and 14 columns.


### Dataset Dimensions

Print a quick snapshot of the dataset's size:

- **`df.shape`** – Tuple of `(rows, columns)`.
- **Total rows** – Number of records, formatted with thousands separators for readability.
- **Total columns** – Number of features/fields in the table.

Useful as a fast sanity check on the dataset's overall scale before diving into deeper profiling or analysis.

In [4]:
print("Shape (rows, columns):", df.shape)
print("Total rows:", f"{df.shape[0]:,}")
print("Total columns:", df.shape[1])

Shape (rows, columns): (580962, 14)
Total rows: 580,962
Total columns: 14


### Column Profile Summary

Build a quick data-quality overview of every column in `df` by assembling a summary DataFrame (`col_info`) with the following metrics:

- **`Column`** – The column name.
- **`Dtype`** – The pandas data type of the column (converted to string for display).
- **`Non-Null`** – Count of non-missing values in the column.
- **`Null`** – Count of missing (`NaN`) values.
- **`Null %`** – Percentage of missing values, rounded to 2 decimals.
- **`Unique`** – Number of distinct non-null values (`dropna=True`).

Finally, `print(col_info.to_string(index=False))` renders the table without the DataFrame index for a clean, aligned view — useful for spotting sparse columns, high-cardinality fields, and type issues at a glance.

In [5]:
col_info = pd.DataFrame({
    "Column": df.columns,
    "Dtype": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Null": df.isna().sum().values,
    "Null %": (df.isna().mean() * 100).round(2).values,
    "Unique": df.nunique(dropna=True).values,
})
print(col_info.to_string(index=False))

             Column   Dtype  Non-Null  Null  Null %  Unique
      TransactionID  object    580962     0    0.00  580962
         CustomerID  object    580962     0    0.00    4997
    TransactionDate  object    580962     0    0.00     639
    TransactionTime  object    580962     0    0.00   71454
    TransactionType  object    580962     0    0.00       9
  TransactionAmount float64    580962     0    0.00  306886
TransactionCurrency  object    569116 11846    2.04       6
          AssetType  object    566487 14475    2.49       5
        CustomerDOB  object    580962     0    0.00    4393
         CustGender  object    580962     0    0.00       2
       CustLocation  object    563766 17196    2.96      23
 CustAccountBalance float64    580962     0    0.00  407472
     WalletActivity   int64    580962     0    0.00       2
  TransactionStatus  object    580962     0    0.00       5


In [6]:
df.head(10)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,None,"5,071.08",1,Completed
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed
5,TXN00000006,CUST00001,2024-12-15,07:10:38,Crypto Buy,74.81,USDT,USDT,2007-04-30,Male,London,"4,172.19",1,Completed
6,TXN00000007,CUST00001,2024-12-15,11:41:13,Cross-Border Payment,441.29,EUR,Traditional,2007-04-30,Male,London,"3,797.09",0,Completed
7,TXN00000008,CUST00001,2024-12-15,19:27:06,Deposit,275.91,EUR,Traditional,2007-04-30,Male,London,"4,031.62",0,Completed
8,TXN00000009,CUST00001,2024-12-16,17:44:56,Deposit,169.85,USD,Traditional,2007-04-30,Male,London,"4,164.10",0,Completed
9,TXN00000010,CUST00001,2024-12-21,08:17:59,P2P Transfer,182.63,USD,None,2007-04-30,Male,London,"4,021.65",0,Completed


In [7]:
df.tail(10)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
580952,TXN00580953,CUST05000,2026-08-11,18:52:37,P2P Transfer,116.75,GBP,Traditional,1971-12-16,Female,Leeds,"1,104.44",0,Completed
580953,TXN00580954,CUST05000,2026-08-13,18:59:52,P2P Transfer,271.57,GBP,Traditional,1971-12-16,Female,Leeds,"1,376.01",0,Completed
580954,TXN00580955,CUST05000,2026-08-16,11:26:04,Withdrawal,393.65,GBP,None,1971-12-16,Female,None,982.36,0,Completed
580955,TXN00580956,CUST05000,2026-08-20,08:34:58,P2P Transfer,429.43,GBP,Traditional,1971-12-16,Female,Leeds,552.93,0,Completed
580956,TXN00580957,CUST05000,2026-08-23,15:17:34,Withdrawal,497.63,GBP,Traditional,1971-12-16,Female,Leeds,55.29,0,Completed
580957,TXN00580958,CUST05000,2026-08-23,17:02:37,Card Payment,58.55,EUR,Traditional,1971-12-16,Female,Leeds,405.16,0,Completed
580958,TXN00580959,CUST05000,2026-08-26,15:37:43,P2P Transfer,265.08,GBP,Traditional,1971-12-16,Female,Leeds,670.23,0,Completed
580959,TXN00580960,CUST05000,2026-08-29,09:16:38,P2P Transfer,709.66,EUR,Traditional,1971-12-16,Female,Leeds,670.23,0,Under Review
580960,TXN00580961,CUST05000,2026-09-02,15:22:40,P2P Transfer,885.57,EUR,Traditional,1971-12-16,Female,Leeds,670.23,0,Failed
580961,TXN00580962,CUST05000,2026-09-06,08:35:28,Card Payment,177.87,GBP,Traditional,1971-12-16,Female,Leeds,492.36,0,Completed


### Numeric Column Summary

- **`df.select_dtypes(include=[np.number])`** – Identify all numeric columns in the DataFrame (ints, floats, etc.), then collect their names into a list (`numeric_cols`).
- **`print(...)`** – Display the list of detected numeric columns for a quick overview.
- **`df[numeric_cols].describe().T`** – Generate descriptive statistics (`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`) for each numeric column. The `.T` (transpose) flips the output so each **row is a column** and each **statistic is a column** — much easier to scan when there are many fields.

In [8]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", numeric_cols)
df[numeric_cols].describe().T

Numeric columns: ['TransactionAmount', 'CustAccountBalance', 'WalletActivity']


,count,mean,std,min,25%,50%,75%,max
TransactionAmount,"580,962.00","4,461.42","17,257.31",0.00,100.31,366.32,"1,177.34","320,512.82"
CustAccountBalance,"580,962.00","24,059.17","71,073.50",0.00,716.88,"2,781.76","8,977.33","1,263,280.33"
WalletActivity,"580,962.00",0.41,0.49,0.00,0.00,0.00,1.00,1.00


### Missing Value Analysis

Build a focused report of missing data across the dataset:

- **`Missing Count`** – Total number of `NaN` values per column (`df.isna().sum()`).
- **`Missing %`** – Share of missing values per column, rounded to 3 decimals (`df.isna().mean() * 100`).
- **`.sort_values("Missing Count", ascending=False)`** – Order columns from most to least missing, so the problem areas surface at the top.

Also prints the **total missing values across the entire dataset** (`df.isna().sum().sum()`) as a single headline number — handy for gauging overall data completeness before deciding on imputation or drop strategies.

In [9]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(3),
}).sort_values("Missing Count", ascending=False)

print("Total missing values in dataset:", df.isna().sum().sum())
missing

Total missing values in dataset: 43517


,Missing Count,Missing %
CustLocation,17196,2.96
AssetType,14475,2.49
TransactionCurrency,11846,2.04
TransactionID,0,0.00
CustomerID,0,0.00
TransactionDate,0,0.00
TransactionTime,0,0.00
TransactionType,0,0.00
TransactionAmount,0,0.00
CustomerDOB,0,0.00


### Duplicate Check

Assess data duplication at three levels to distinguish real data-quality issues from expected repetition:

- **`df.duplicated().sum()`** → `duplicate_rows` – Count of fully duplicated rows (every column identical). Any value > 0 signals redundant records worth investigating.
- **`df["TransactionID"].duplicated().sum()`** → `duplicate_txnids` – Count of repeated `TransactionID` values. Since `TransactionID` should be a **unique key**, any duplicates here indicate a data integrity problem.
- **`df["CustomerID"].duplicated().sum()`** → `duplicate_custids` – Count of repeated `CustomerID` values. This is **expected** in a transactional table (one customer → many transactions), so a high count is normal and not a red flag.

Together, these three checks separate genuine duplication issues (rows / transaction IDs) from legitimate one-to-many relationships (customer IDs).

In [10]:
duplicate_rows = df.duplicated().sum()
duplicate_txnids = df["TransactionID"].duplicated().sum()
duplicate_custids = df["CustomerID"].duplicated().sum()

print(f"Fully duplicated rows: {duplicate_rows}")
print(f"Duplicated TransactionIDs: {duplicate_txnids}")
print(f"Duplicated CustomerIDs (expected — one customer, many txns): {duplicate_custids:,}")

Fully duplicated rows: 0
Duplicated TransactionIDs: 0
Duplicated CustomerIDs (expected — one customer, many txns): 575,965


### Customer & Transaction Volume

Compute a few high-level engagement metrics to understand the structure of the transactional data:

- **`df["CustomerID"].nunique()`** – Number of distinct customers in the dataset.
- **`len(df)`** – Total number of transactions (rows).
- **`len(df) / df["CustomerID"].nunique()`** – Average number of transactions per customer, rounded to 2 decimals — a quick indicator of purchase frequency / engagement depth.

In [11]:
print("Unique customers:", df["CustomerID"].nunique())
print("Total transactions:", len(df))
print("Avg transactions per customer:",
      round(len(df) / df["CustomerID"].nunique(), 2))

Unique customers: 4997
Total transactions: 580962
Avg transactions per customer: 116.26


### Cardinality Check (Unique Values per Column)

Count distinct values in each key categorical and identifier column to understand cardinality and confirm expected uniqueness:

| Column | Expected Cardinality |
|---|---|
| **`TransactionID`** | Should equal total row count — unique primary key per transaction. |
| **`CustomerID`** | Number of distinct customers (one-to-many with transactions). |
| **`TransactionType`** | Small set (e.g., Deposit, Withdrawal, Transfer). |
| **`TransactionCurrency`** | Small set of currency codes (e.g., USD, NGN, EUR). |
| **`AssetType`** | Small set of tradable asset classes. |
| **`CustLocation`** | Moderate cardinality (cities/regions). |
| **`TransactionStatus`** | Small set (e.g., Success, Failed, Pending). |
| **`CustGender`** | Very low cardinality (typically 2–3 values). |

Useful for:
- **Verifying key integrity** — `TransactionID` uniqueness confirms no primary-key violations.
- **Spotting data quality issues** — unexpected high/low cardinality (e.g., typos, inconsistent casing, mixed encodings) often shows up here.
- **Planning encoding** — informs whether to use one-hot, ordinal, or target encoding downstream.

In [12]:
print("Unique TransactionID:", df["TransactionID"].nunique())
print("Unique CustomerID:", df["CustomerID"].nunique())
print("Unique TransactionType:", df["TransactionType"].nunique())
print("Unique TransactionCurrency:", df["TransactionCurrency"].nunique())
print("Unique AssetType:", df["AssetType"].nunique())
print("Unique CustLocation:", df["CustLocation"].nunique())
print("Unique TransactionStatus:", df["TransactionStatus"].nunique())
print("Unique CustGender:", df["CustGender"].nunique())

Unique TransactionID: 580962
Unique CustomerID: 4997
Unique TransactionType: 9
Unique TransactionCurrency: 6
Unique AssetType: 5
Unique CustLocation: 23
Unique TransactionStatus: 5
Unique CustGender: 2


### Date Parsing & Range Inspection

Convert date columns from strings to proper `datetime64` types, then inspect their ranges to validate data quality and understand the temporal scope.

**Parsing:**
- **`TransactionDate`** → datetime — when the transaction occurred.
- **`CustomerDOB`** → datetime — customer's date of birth.

**Transaction date range:**
- Reports the **earliest** and **latest** transaction dates, plus the **total span in days** — establishes the observation window and confirms dates fall within expected bounds (catches outliers like dates in the future or far past).

**Customer DOB range:**
- Reports the **earliest** and **latest** birth dates — useful for sanity-checking age distribution and flagging impossible DOBs (e.g., pre-1900 or future dates).

**`TransactionTime` sample:**
- Prints the first 3 raw values as a preview. Since time is stored separately from date, this confirms its format (e.g., `"HH:MM:SS"` string) before any parsing or feature engineering.

In [13]:
df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])
df["CustomerDOB"] = pd.to_datetime(df["CustomerDOB"])

print("Earliest TransactionDate:", df["TransactionDate"].min())
print("Latest   TransactionDate:", df["TransactionDate"].max())
print("Date span (days):", (df["TransactionDate"].max() - df["TransactionDate"].min()).days)

print("\nEarliest CustomerDOB:", df["CustomerDOB"].min())
print("Latest   CustomerDOB:", df["CustomerDOB"].max())

print("\nTransactionTime sample:", df["TransactionTime"].head(3).tolist())

Earliest TransactionDate: 2024-12-13 00:00:00
Latest   TransactionDate: 2026-09-12 00:00:00
Date span (days): 638

Earliest CustomerDOB: 1955-10-08 00:00:00
Latest   CustomerDOB: 2008-09-12 00:00:00

TransactionTime sample: ['10:03:23', '17:15:53', '12:26:16']


In [14]:
txns_per_cust = df.groupby("CustomerID").size()

print("Transactions per customer — summary:")
print(txns_per_cust.describe().round(2))

Transactions per customer — summary:
count   4,997.00
mean      116.26
std        95.47
min         1.00
25%        16.00
50%       128.00
75%       177.00
max       487.00
dtype: float64


In [15]:
top_cust = (df.groupby("CustomerID")
              .agg(txn_count=("TransactionID", "count"),
                   total_amount=("TransactionAmount", "sum"))
              .sort_values("total_amount", ascending=False)
              .head(10))
top_cust

,txn_count,total_amount
CustomerID,,
CUST00394,378,"6,648,767.52"
CUST02532,424,"6,316,046.39"
CUST00611,424,"6,099,789.96"
CUST03051,369,"5,995,617.16"
CUST01335,448,"5,992,918.65"
CUST02048,453,"5,933,027.01"
CUST02769,405,"5,910,265.40"
CUST03636,352,"5,844,867.16"
CUST04653,477,"5,830,904.28"


In [16]:
print("CustAccountBalance — summary:")
print(df["CustAccountBalance"].describe().round(2))
print()
print("Negative balance rows:", (df["CustAccountBalance"] < 0).sum())
print("Zero balance rows:", (df["CustAccountBalance"] == 0).sum())

CustAccountBalance — summary:
count     580,962.00
mean       24,059.17
std        71,073.50
min             0.00
25%           716.88
50%         2,781.76
75%         8,977.33
max     1,263,280.33
Name: CustAccountBalance, dtype: float64

Negative balance rows: 0
Zero balance rows: 38


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580962 entries, 0 to 580961
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   TransactionID        580962 non-null  object        
 1   CustomerID           580962 non-null  object        
 2   TransactionDate      580962 non-null  datetime64[ns]
 3   TransactionTime      580962 non-null  object        
 4   TransactionType      580962 non-null  object        
 5   TransactionAmount    580962 non-null  float64       
 6   TransactionCurrency  569116 non-null  object        
 7   AssetType            566487 non-null  object        
 8   CustomerDOB          580962 non-null  datetime64[ns]
 9   CustGender           580962 non-null  object        
 10  CustLocation         563766 non-null  object        
 11  CustAccountBalance   580962 non-null  float64       
 12  WalletActivity       580962 non-null  int64         
 13  TransactionSta